# Steerable Pyramid

**Utility Functions**

In [ ]:
import numpy as np

def is_even(numbers):
    """
    Check if numbers are even.

    Parameters:
    - numbers: int, float, or array-like

    Returns:
    - result: bool or array-like
    """
    return (numbers == (numbers // 2) * 2)


def is_odd(numbers):
    """
    Check if numbers are odd.

    Parameters:
    - numbers: int, float, or array-like

    Returns:
    - result: bool or array-like
    """
    return 1 - (numbers == (numbers // 2) * 2)


def max2(matrix):
    """
    Compute the maximum value of a 2D matrix.
    """
    return np.max(matrix)

def min2(matrix):
    """
    Compute the minimum value of a 2D matrix.
    """
    return np.min(matrix)

def mean2(matrix):
    """
    Compute the mean value of a 2D matrix.
    """
    return np.mean(matrix)


def max_level(dims, bandwidth):
    """
    Compute the maximum number of levels for a steerable pyramid.

    Parameters:
    - dims: tuple, image size (M, N).
    - bandwidth: float, spatial frequency bandwidth in octaves.

    Returns:
    - lev: int, maximum number of levels.
    """
    return int(np.floor((np.log2(min(dims)) - 2) / bandwidth))




def mse(A, B):
    """
    Compute the mean-squared-error between two matrices/images.

    Parameters:
    - A, B: 2D arrays, the matrices to compare.

    Returns:
    - result: float, the mean-squared-error.
    """
    return np.mean(np.abs(A - B) ** 2)



import numpy as np

def log0(x, base=np.e, zero_val=0):
    """
    Compute logarithm with a custom base and value for zero inputs.

    Parameters:
    - x: array-like, input values.
    - base: float, the base of the logarithm (default: natural log).
    - zero_val: float, value to assign for log(0) (default: 0).

    Returns:
    - y: array-like, logarithm values.
    """
    y = np.full_like(x, zero_val, dtype=np.float64)
    positive_indices = x > 0
    y[positive_indices] = np.log(x[positive_indices]) / np.log(base)
    return y



def log_raised_cos(r, CtrFreq, bandwidth):
    """
    Logarithmic raised cosine filter function.
    """
    Rarg = (np.pi / bandwidth) * np.log2(np.maximum(r / (CtrFreq / np.pi), 1e-10))
    y = np.sqrt(0.5 * (np.cos(Rarg) + 1))
    y = np.where((-np.pi < Rarg) & (Rarg < np.pi), y, 0)
    return y

def log_raised_cos_hi(r, CtrFreq, bandwidth):
    """
    High-pass raised cosine filter.
    """
    CtrFreq *= 2 ** bandwidth
    Rarg = (np.pi / bandwidth) * np.log2(np.maximum(r / (CtrFreq / np.pi), 1e-10))
    y = np.sqrt(0.5 * (np.cos(Rarg) + 1))
    y = np.where((-np.pi < Rarg) & (Rarg < 0.0), y, 0)
    y = np.where(Rarg >= 0.0, 1, y)
    return y

def log_raised_cos_lo(r, CtrFreq, bandwidth):
    """
    Low-pass raised cosine filter.
    """
    CtrFreq /= 2 ** bandwidth
    Rarg = (np.pi / bandwidth) * np.log2(np.maximum(r / (CtrFreq / np.pi), 1e-10))
    y = np.sqrt(0.5 * (np.cos(Rarg) + 1))
    y = np.where((0.0 < Rarg) & (Rarg < np.pi), y, 0)
    y = np.where(Rarg <= 0.0, 1, y)
    return y


**core func**

In [ ]:
def access_steer_band(pyr, pind, num_orientations, level, orientation):
    """
    Retrieve a specific band (level, orientation) from a steerable pyramid.

    Parameters:
    - pyr: 1D array, the image transform (flattened pyramid coefficients).
    - pind: list of tuples, sizes of each band in the pyramid.
    - num_orientations: int, number of orientation subbands.
    - level: int, desired level/scale (1-based).
    - orientation: int, desired orientation (1-based).

    Returns:
    - res: 2D array, the requested band.
    """
    b = (level - 1) * num_orientations + orientation + 1  # 1-based indexing
    indices = pyr_band_indices(pind, b)
    band_coeffs = pyr[indices]
    return band_coeffs.reshape(pind[b - 1])  # Reshape to band dimensions


import numpy as np

def build_steer_bands(im, freq_resps):
    """
    Build multiscale subbands of a steerable pyramid.

    Parameters:
    - im: 2D array, the input image.
    - freq_resps: 2D array, filter frequency responses (flattened).

    Returns:
    - pyr: 1D array, flattened pyramid coefficients.
    - pind: list of tuples, sizes of each band.
    """
    M, N = im.shape
    total_size = M * N
    numbands = freq_resps.shape[0] // total_size
    pind = [(M, N)] * numbands

    # Fourier transform of the input image
    fourier = np.fft.fftshift(np.fft.fft2(im))

    # Initialize the pyramid
    pyr = np.zeros(freq_resps.shape, dtype=np.float64)

    for b in range(1, numbands + 1):  # 1-based indexing
        indices = pyr_band_indices(pind, b)
        freq_resp = freq_resps[indices].reshape(pind[b - 1])  # Reshape filter response
        band = np.real(np.fft.ifft2(np.fft.ifftshift(fourier * freq_resp)))
        pyr[indices] = band.flatten()

    return pyr, pind


def build_quad_bands(im, freq_resps_imag, freq_resps_real):
    """
    Build quadrature pair multiscale subbands.

    Parameters:
    - im: 2D array, the input image.
    - freq_resps_imag: 2D array, imaginary part of filter frequency responses.
    - freq_resps_real: 2D array, real part of filter frequency responses.

    Returns:
    - pyr: 1D array, flattened pyramid coefficients (complex values).
    - pind: list of tuples, sizes of each band.
    """
    if freq_resps_imag.shape != freq_resps_real.shape:
        raise ValueError("freq_resps_imag and freq_resps_real must have the same shape")

    # Build steerable pyramid bands for real and imaginary parts
    pyr_imag, pind = build_steer_bands(im, freq_resps_imag)
    pyr_real, _ = build_steer_bands(im, freq_resps_real)

    # Combine real and imaginary parts to form complex coefficients
    pyr = pyr_real + 1j * pyr_imag

    return pyr, pind


def make_quad_frs(dims, num_levels, num_orientations, bandwidth):
    """
    Generate quadrature pairs of frequency responses.

    Parameters:
    - dims: tuple, image size (M, N).
    - num_levels: int, number of levels/scales.
    - num_orientations: int, number of orientation subbands.
    - bandwidth: float, spatial frequency bandwidth in octaves.

    Returns:
    - freq_resps_imag: array, imaginary part of frequency responses.
    - freq_resps_real: array, real part of frequency responses.
    - pind: list of tuples, sizes of each resulting subband image.
    """
    if is_odd(num_orientations):
        raise ValueError("num_orientations must be even in make_quad_frs.")

    freq_resps_imag, pind = make_steer_frs(dims, num_levels, num_orientations, bandwidth)
    freq_resps_real = np.copy(freq_resps_imag)

    for b in range(1, num_levels * num_orientations + 1):  # Exclude low and high bands
        indices = pyr_band_indices(pind, b)
        freq_resps_real[indices] = np.abs(freq_resps_real[indices])

    # Zero out low and high bands in imaginary responses
    low_indices = pyr_band_indices(pind, 1)
    freq_resps_imag[low_indices] = 0
    high_indices = pyr_band_indices(pind, num_levels * num_orientations + 2)
    freq_resps_imag[high_indices] = 0

    return freq_resps_imag, freq_resps_real, pind


def make_steer_frs(dims, num_levels, num_orientations, bandwidth):
    """
    Generate frequency responses for a steerable pyramid.

    Parameters:
    - dims: tuple, image size (M, N).
    - num_levels: int, number of levels/scales.
    - num_orientations: int, number of orientation subbands.
    - bandwidth: float, spatial frequency bandwidth in octaves.

    Returns:
    - result: 1D array, vector containing all subband frequency responses.
    - pind: list of tuples, sizes of each resulting subband image.
    """
    numbands = num_levels * num_orientations + 2
    pind = [dims] * numbands

    p = num_orientations - 1
    const = np.sqrt((2 ** (2 * p) * np.math.factorial(p) ** 2) / (np.math.factorial(2 * p) * (p + 1)))

    wx, wy = np.meshgrid(np.fft.fftfreq(dims[1]), np.fft.fftfreq(dims[0]))
    r = np.sqrt(wx ** 2 + wy ** 2)
    theta = np.arctan2(wy, wx)

    result = np.zeros(numbands * np.prod(dims))

    # Bandpass bands
    for level in range(1, num_levels + 1):
        for orientation in range(num_orientations):
            theta_offset = orientation * np.pi / num_orientations
            ctr_freq = np.pi / (2 ** (level * bandwidth))
            band = (1j ** p) * const * (np.cos(theta - theta_offset) ** p) * log_raised_cos(r, ctr_freq, bandwidth)
            b = (level - 1) * num_orientations + (orientation + 1)
            indices = pyr_band_indices(pind, b)
            result[indices] = band.flatten()

    # High band
    high_band = log_raised_cos_hi(r, np.pi / (2 ** bandwidth), bandwidth)
    result[:dims[0] * dims[1]] = high_band.flatten()

    # Low band
    low_band = log_raised_cos_lo(r, np.pi / (2 ** (bandwidth * num_levels)), bandwidth)
    result[-dims[0] * dims[1]:] = low_band.flatten()

    return result, pind




def view_bands(pyr, pind, wait=1, value_range='auto1'):
    """
    View the subbands of the transform.

    Parameters:
    - pyr: 1D array, the image transform (flattened pyramid coefficients).
    - pind: list of tuples, sizes of each band in the pyramid.
    - wait: float, time to wait between displaying successive subband images.
    - value_range: str or tuple, range of values for scaling ('auto1' or (min, max)).
    """
    numbands = len(pind)
    for b in range(1, numbands + 1):
        indices = pyr_band_indices(pind, b)
        band = pyr[indices].reshape(pind[b - 1])
        display_image(band, value_range=value_range)
        plt.pause(wait)


**rebuild**

In [ ]:
def pyr_hi(pyr, pind):
    """
    Access the highpass residual band from a steerable pyramid.

    Parameters:
    - pyr: 1D array, the image transform (flattened pyramid coefficients).
    - pind: list of tuples, sizes of each band in the pyramid.

    Returns:
    - res: 2D array, highpass residual band.
    """
    return pyr_band(pyr, pind, 1)


def recon_steer_bands(pyr, freq_resps, pind):
    """
    Reconstruct an image from the subband transform.

    Parameters:
    - pyr: 1D array, the image transform (flattened pyramid coefficients).
    - freq_resps: 2D array, filter frequency responses.
    - pind: list of tuples, sizes of each band in the pyramid.

    Returns:
    - result: 2D array, reconstructed image.
    """
    M, N = pind[0]
    result = np.zeros((M, N))

    numbands = len(pind)
    for b in range(1, numbands + 1):
        indices = pyr_band_indices(pind, b)
        freq_resp = freq_resps[indices].reshape(pind[b - 1])
        band = pyr[indices].reshape(pind[b - 1])
        freq_band = np.fft.fftshift(np.fft.fft2(band))
        result += np.real(np.fft.ifft2(np.fft.ifftshift(freq_band * np.conj(freq_resp))))

    return result


**normalization**

In [ ]:
def norm_energies(pyr, pind, num_orientations, sigma=0):
    """
    Compute contrast-normalized local energy responses from a quadrature subband transform.

    Parameters:
    - pyr: 1D array, the image transform (flattened pyramid coefficients).
    - pind: list of tuples, sizes of each band in the pyramid.
    - num_orientations: int, number of orientation subbands.
    - sigma: float, semi-saturation constant for contrast normalization.

    Returns:
    - result: 1D array, normalized energy responses.
    """
    dims = pind[0]
    num_levels = (len(pind) - 2) // num_orientations

    result = np.zeros_like(pyr)
    energies = np.abs(pyr) ** 2
    normalizers = np.zeros((num_levels + 1) * np.prod(dims))
    norm_pind = [dims] * (num_levels + 1)

    # Highpass normalizer
    hi = blur(pyr_hi(energies, pind))
    norm_indices = pyr_band_indices(norm_pind, 1)
    normalizers[norm_indices] = hi.flatten()

    # Compute normalizers per level and orientation
    for level in range(1, num_levels + 1):
        for orientation in range(1, num_orientations + 1):
            b = (level - 1) * num_orientations + orientation + 1
            pyr_indices = pyr_band_indices(pind, b)
            norm_indices = pyr_band_indices(norm_pind, level + 1)
            normalizers[norm_indices] += energies[pyr_indices]

    # Normalize each bandpass energy band
    for level in range(1, num_levels):
        for orientation in range(1, num_orientations + 1):
            b = (level - 1) * num_orientations + orientation + 1
            pyr_indices = pyr_band_indices(pind, b)
            norm_indices1 = pyr_band_indices(norm_pind, level)
            norm_indices2 = pyr_band_indices(norm_pind, level + 1)
            norm_indices3 = pyr_band_indices(norm_pind, level + 2)
            result[pyr_indices] = energies[pyr_indices] / (
                normalizers[norm_indices1] +
                normalizers[norm_indices2] +
                normalizers[norm_indices3] +
                sigma ** 2
            )

    return result



def norm_energies_by_freq(pyr, pind, num_orientations, sigma=0, which_level=None):
    """
    Compute contrast-normalized local energy responses for specific levels of a pyramid.

    Parameters:
    - pyr: 1D array, the image transform (flattened pyramid coefficients).
    - pind: list of tuples, sizes of each band in the pyramid.
    - num_orientations: int, number of orientation subbands.
    - sigma: float, semi-saturation constant for contrast normalization.
    - which_level: list, levels to process.

    Returns:
    - result: 1D array, normalized energy responses.
    """
    if which_level is None:
        raise ValueError("which_level must be specified.")

    dims = pind[0]
    num_levels = (len(pind) - 2) // num_orientations

    result = np.zeros_like(pyr)
    energies = np.abs(pyr) ** 2
    normalizers = np.zeros((num_levels + 1) * np.prod(dims))
    norm_pind = [dims] * (num_levels + 1)

    # Highpass normalizer
    hi = blur(pyr_hi(energies, pind))
    norm_indices = pyr_band_indices(norm_pind, 1)
    normalizers[norm_indices] = hi.flatten()

    # Compute normalizers for specific levels
    for level in which_level:
        for orientation in range(1, num_orientations + 1):
            b = (level - 1) * num_orientations + orientation + 1
            pyr_indices = pyr_band_indices(pind, b)
            norm_indices = pyr_band_indices(norm_pind, level + 1)
            normalizers[norm_indices] += energies[pyr_indices]

    # Normalize as in normEnergies
    # Similar normalization logic as normEnergies


def norm_energies_by_ori(pyr, pind, num_orientations, sigma=0, which_ori=None):
    """
    Compute contrast-normalized local energy responses for a specific orientation.

    Parameters:
    - pyr: 1D array, the image transform (flattened pyramid coefficients).
    - pind: list of tuples, sizes of each band in the pyramid.
    - num_orientations: int, number of orientation subbands.
    - sigma: float, semi-saturation constant for the contrast normalization.
    - which_ori: int, orientation to process (1-based index).

    Returns:
    - result: 1D array, normalized energy responses.
    """
    if which_ori is None:
        raise ValueError("which_ori must be specified.")

    dims = pind[0]
    num_levels = (len(pind) - 2) // num_orientations

    result = np.zeros_like(pyr)
    energies = np.abs(pyr) ** 2
    normalizers = np.zeros((num_levels + 1) * np.prod(dims))
    norm_pind = [dims] * (num_levels + 1)

    # Compute highpass normalizer (spatial blurring because there is no quadrature pair)
    hi = blur(pyr_hi(energies, pind))
    norm_indices = pyr_band_indices(norm_pind, 1)
    normalizers[norm_indices] = hi.flatten()

    # Compute normalizers for the specified orientation
    for level in range(1, num_levels + 1):
        b = (level - 1) * num_orientations + which_ori
        pyr_indices = pyr_band_indices(pind, b)
        norm_indices = pyr_band_indices(norm_pind, level + 1)
        normalizers[norm_indices] += energies[pyr_indices]

    # Normalize each bandpass energy band, except the lowest spatial frequency
    for level in range(1, num_levels):
        for orientation in range(1, num_orientations + 1):
            b = (level - 1) * num_orientations + orientation
            pyr_indices = pyr_band_indices(pind, b)
            norm_indices1 = pyr_band_indices(norm_pind, level)
            norm_indices2 = pyr_band_indices(norm_pind, level + 1)
            norm_indices3 = pyr_band_indices(norm_pind, level + 2)
            result[pyr_indices] = energies[pyr_indices] / (
                normalizers[norm_indices1] +
                normalizers[norm_indices2] +
                normalizers[norm_indices3] +
                sigma ** 2
            )

    return result


**display**

In [ ]:
import matplotlib.pyplot as plt

def display_image(im, value_range='auto1', nshades=256, style='gray'):
    """
    Display a matrix as an image.

    Parameters:
    - im: 2D or 3D array, input image or images.
    - value_range: str or tuple, range of values for scaling ('auto1', 'auto2', or (min, max)).
    - nshades: int, number of gray shades (ignored for RGB images).
    - style: str, display style ('gray', 'rgb', etc.).

    Returns:
    - None
    """
    if value_range == 'auto1' or value_range == 'auto':
        vmin, vmax = im.min(), im.max()
    elif value_range == 'auto2':
        mean = im.mean()
        vmax = max(mean - im.min(), im.max() - mean)
        vmin, vmax = mean - vmax, mean + vmax
    else:
        vmin, vmax = value_range

    if style == 'gray':
        plt.imshow(im, cmap='gray', vmin=vmin, vmax=vmax)
    elif style == 'rgb':
        plt.imshow(im, vmin=vmin, vmax=vmax)
    else:
        raise ValueError(f"Unsupported style: {style}")

    plt.axis('off')
    plt.show()


def example_steerable_pyramid():
    """
    Example usage of steerable pyramid functions.
    """
    # Define parameters
    dims = (64, 64)
    num_levels = 5
    num_orientations = 4
    bandwidth = 0.5

    # Generate frequency responses
    freq_resps, pind = make_steer_frs(dims, num_levels, num_orientations, bandwidth)

    # Display a band
    band = access_steer_band(freq_resps, pind, num_orientations, 1, 1)
    display_image(band, style='gray')

    # Check tiling
    tile = np.zeros(dims)
    tile += np.abs(pyr_low(freq_resps, pind)) ** 2
    tile += np.abs(pyr_hi(freq_resps, pind)) ** 2
    for lev in range(1, num_levels + 1):
        for orientation in range(1, num_orientations + 1):
            band = access_steer_band(freq_resps, pind, num_orientations, lev, orientation)
            tile += np.abs(band) ** 2

    display_image(tile, value_range=(0.999, 1.001), style='gray')



Access Data

In [ ]:
from google.colab import files
uploaded = files.upload()

Install Dependencies

In [ ]:
!pip install pyrtools

Import Libraries and Set Up Constants

In [ ]:
import os
import numpy as np
import h5py
from scipy.ndimage import zoom
from skimage.transform import resize
from pyrtools import SteerablePyramidFreq
from scipy.io import loadmat

# Constants
interp_img_size = 714
background_size = 1024
img_scaling = 0.5
pyramid_folder = '/content/pyramid_expand'
os.makedirs(pyramid_folder, exist_ok=True)


Load Stimuli

In [ ]:
stim_folder = '/content/stimuli/'
stim_filename = os.path.join(stim_folder, 'nsd_stimuli.hdf5')

with h5py.File(stim_filename, 'r') as f:
    stim_info = f['/imgBrick/']
    img_size_x, img_size_y, num_imgs = stim_info.shape[1:4]

nsd_folder = '/content/nsd/'
nsd_design_filename = os.path.join(nsd_folder, 'nsd_expdesign.mat')
nsd_design = loadmat(nsd_design_filename)
all_imgs = nsd_design['sharedix'].flatten()


Preprocess and Process Images

Save Results